In [4]:
import pandas as pd
import numpy as np
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('statsbomb_euro2020.db')

# To list all tables in the database
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in the database:")
print(tables)

# Assign each table to a separate dataframe
for table_name in tables['name']:
    try:
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        # Create a variable with the table name as the dataframe name
        globals()[f"{table_name}_df"] = df
        print(f"Loaded table '{table_name}' into dataframe '{table_name}_df'")
    except Exception as e:
        print(f"Could not read table {table_name}: {e}")

# Don't forget to close the connection when done
conn.close()


Tables in the database:
              name
0          lineups
1  sqlite_sequence
2       frames_360
3          matches
4           events
Loaded table 'lineups' into dataframe 'lineups_df'
Loaded table 'sqlite_sequence' into dataframe 'sqlite_sequence_df'
Loaded table 'frames_360' into dataframe 'frames_360_df'
Loaded table 'matches' into dataframe 'matches_df'
Loaded table 'events' into dataframe 'events_df'


In [16]:
# Create a DataFrame to track coverage and counts for each match_id in matches_df

# Get all unique match_ids from matches_df
all_match_ids = matches_df['match_id'].unique()

# Prepare lists to store results
coverage_data = []

for match_id in all_match_ids:
    in_frames = match_id in frames_360_df['match_id'].values
    in_lineups = match_id in lineups_df['match_id'].values
    in_events = match_id in events_df['match_id'].values
    in_matches = match_id in matches_df['match_id'].values  # will always be True

    frames_count = frames_360_df[frames_360_df['match_id'] == match_id].shape[0] if in_frames else 0
    lineups_count = lineups_df[lineups_df['match_id'] == match_id].shape[0] if in_lineups else 0
    events_count = events_df[events_df['match_id'] == match_id].shape[0] if in_events else 0
    matches_count = matches_df[matches_df['match_id'] == match_id].shape[0] if in_matches else 0

    coverage_data.append({
        'match_id': match_id,
        'in_frames_360': in_frames,
        'frames_360_count': frames_count,
        'in_lineups': in_lineups,
        'lineups_count': lineups_count,
        'in_events': in_events,
        'events_count': events_count,
        'in_matches': in_matches,
        'matches_count': matches_count
    })

coverage_df = pd.DataFrame(coverage_data)

# Show the coverage DataFrame

In [25]:
# For each match_id in coverage_df, grab the corresponding row from matches_df
# and add the key columns to coverage_df

# List of columns to add from matches_df
matches_columns = [
    'match_date', 'kick_off', 'competition', 'season', 'home_team', 'away_team',
    'home_score', 'away_score', 'match_status', 'match_status_360', 'last_updated',
    'last_updated_360', 'match_week', 'competition_stage', 'stadium', 'referee',
    'home_managers', 'away_managers', 'data_version', 'shot_fidelity_version', 'xy_fidelity_version'
]

# Merge the relevant columns from matches_df into coverage_df based on match_id
coverage_df = coverage_df.merge(
    matches_df[['match_id'] + matches_columns],
    on='match_id',
    how='left'
)


In [6]:
pd.set_option('display.max_columns', None)

In [27]:
frames_360_df.to_csv('frames_360_df.csv', index=False)
lineups_df.to_csv('lineups_df.csv', index=False)
events_df.to_csv('events_df.csv', index=False)
matches_df.to_csv('matches_df.csv', index=False)
